# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  
**Ciara Graves**

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
# imported the data to my gdrive, so need to mount
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd
# changed path to path in my drive
PATH = '/content/drive/MyDrive/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [3]:
# TODO (hint: .str.lower().str.contains('pop'))
# filter for any rows containing 'pop'
pop_tarts = cat[cat['product_name'].str.lower().str.contains('pop')]

# Print count and matching rows
print("Number of Pop-Tart products:", len(pop_tarts))
print(pop_tarts)

Number of Pop-Tart products: 4
             sku          product_name         category subcategory  \
0          PT-12  pop-tarts strawberry  Breakfast Foods         NaN   
1     SKU#459812   Strawberry Pop-Tart        Breakfast   Pop-Tarts   
2  POPTART-STRAW  Pop-Tarts Strawberry        Breakfast         NaN   
3  PT_STRAWBERRY  pop-tarts strawberry        breakfast   Pop-Tarts   

   unit_cost           supplier package_size  
0       1.93  Johnson & Johnson       4-pack  
1       2.26           Roku Inc          4pk  
2       2.11           Sara Lee       6 pack  
3        NaN         Campbell's       4 pack  


**Q1 Explanation:** Frim our filtering, we see that there are 4 different Pop-Tart rows in our data. All appear to be strawberry, however their supplier, package size, and cost differ.

### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [4]:
# TODO
# Def'n strawberry poptarts
strawberry = cat[
    cat['product_name'].str.contains('pop', case=False, na=False) &
    cat['product_name'].str.contains('strawberry', case=False, na=False)
]

# Ouput SKUs & names
print(strawberry[['sku', 'product_name']])

# For good measure, look for any other Pop-Tarts in the data (a lil redunant)
# Find all non-strawberry Pop-Tarts (pop tarts not containing strawberry)
non_strawberry = cat[
    cat['product_name'].str.contains('pop', case=False, na=False) &
    ~cat['product_name'].str.contains('strawberry', case=False, na=False)]

print("\nNon-strawberry Pop-Tarts:")
print(non_strawberry[['sku', 'product_name']])

# non-straw Gave empty df, search for ANY Pop-tarts
all_pop_tarts = cat[cat['product_name'].str.contains('pop', case=False, na=False)]
print("\nAll Pop-tarts:")
print(all_pop_tarts[['sku', 'product_name']])

             sku          product_name
0          PT-12  pop-tarts strawberry
1     SKU#459812   Strawberry Pop-Tart
2  POPTART-STRAW  Pop-Tarts Strawberry
3  PT_STRAWBERRY  pop-tarts strawberry

Non-strawberry Pop-Tarts:
Empty DataFrame
Columns: [sku, product_name]
Index: []

All Pop-tarts:
             sku          product_name
0          PT-12  pop-tarts strawberry
1     SKU#459812   Strawberry Pop-Tart
2  POPTART-STRAW  Pop-Tarts Strawberry
3  PT_STRAWBERRY  pop-tarts strawberry


**Pop-Tart rows that are a different product:**

There are 4 different SKUs denoting strawberry Pop-Tarts. Beyond that, there are no other, non-strawberry, Pop-Tarts in the data. This makes sense becuase Q1 told us that there were 4 different types of Pop-Tarts, it's nice to see that result verified.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [5]:
# TODO
# Map variant SKUs to one canonical SKU
sku_map = {
    'PT-12': 'PT-12',
    'SKU#459812': 'PT-12',
    'POPTART-STRAW': 'PT-12',
    'PT_STRAWBERRY': 'PT-12'
}

# Apply mapping, keeping unmapped SKUs unchanged
cat['canonical_sku'] = cat['sku'].replace(sku_map)

# Check results
print(cat[['sku', 'canonical_sku', 'product_name']])

              sku  canonical_sku            product_name
0           PT-12          PT-12    pop-tarts strawberry
1      SKU#459812          PT-12     Strawberry Pop-Tart
2   POPTART-STRAW          PT-12    Pop-Tarts Strawberry
3   PT_STRAWBERRY          PT-12    pop-tarts strawberry
4          WTR-24         WTR-24      BOTTLED WATER 24PK
5       WATER24PK      WATER24PK   Bottled Water 24-Pack
6        WTR_24PK       WTR_24PK         Water - 24 Pack
7        BATT-100       BATT-100        BATTERIES AA 4PK
8         BAT-AA4        BAT-AA4     AA Batteries 4-Pack
9     BATT_AA_4PK    BATT_AA_4PK     AA Batteries 4-Pack
10       FLASH-01       FLASH-01          FLASHLIGHT LED
11      LED-FLASH      LED-FLASH          FLASHLIGHT LED
12        BEER-24        BEER-24    Light Beer (24-pack)
13       LTBEER24       LTBEER24    Light Beer (24-pack)
14        BG-MONO        BG-MONO     Monopoly Board Game
15    MONOPOLY-01    MONOPOLY-01     Monopoly Board Game
16      GAME-MONO      GAME-MON

**Q3 Explanation:** Here, I used `.replace()` instead of `.map()` so I could preserve the unmapped SKUs. This allows for us to create the dictionary as desired.

### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [6]:
# TODO
# Show categories before cleaning
print("Before normalization:")
print(cat['category'].value_counts(dropna=False))

# Strip whitespace and title-case
cat['category'] = cat['category'].str.strip().str.title()

# Show categories after cleaning
print("\nAfter normalization:")
print(cat['category'].value_counts(dropna=False))

# Count distinct categories
print("\nDistinct categories:", cat['category'].nunique())

Before normalization:
category
Emergency              5
Emergency Supplies     3
EMERGENCY              3
ENTERTAINMENT          3
GROCERY                3
Breakfast              2
entertainment          2
household              2
ELECTRONICS            2
health                 2
Baby                   2
Electronics            1
ALCOHOL                1
Alcoholic Beverages    1
breakfast              1
Breakfast Foods        1
Groceries              1
Grocery                1
electronics            1
snacks                 1
grocery                1
HOUSEHOLD              1
Snacks                 1
Household              1
baby                   1
Baby Products          1
emergency              1
Name: count, dtype: int64

After normalization:
category
Emergency              9
Entertainment          5
Grocery                5
Electronics            4
Household              4
Emergency Supplies     3
Breakfast              3
Baby                   3
Snacks                 2
Health      

**Q4 Explanation:** The following pairs are still seperate labels for the same category after normalization: Emergency and Emergency Supplies, Breakfast and Breakfast Foods, Alcohol and Alcoholic Beverages, Grocery and Groceries, and Baby and Baby Products. I could further normalize the categories by looking for the starting 4 or so same letters to combine on, however that could pose future issues by blinding blending categories. If we need to combine any of these pairs, like Breakfast and Breakfast Foods, it is the safest option to do so on an individual, intentional basis.

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [7]:
# TODO
# Need to normalize product names
cat['normalized_name'] = (
    cat['product_name']
    .str.lower()
    .str.replace(r'[^\w\s]', '', regex=True)
    .str.split()
    .apply(lambda x: ' '.join(sorted(x)))
)

# Group by normalized name and count unique SKUs
sku_counts = cat.groupby('normalized_name')['sku'].nunique()

# Keep groups with more than one SKU
candidates = sku_counts[sku_counts > 1]

# Show matching rows
matches = cat[cat['normalized_name'].isin(candidates.index)]
# Ouput
print(matches[['sku', 'product_name', 'normalized_name']])

              sku           product_name      normalized_name
0           PT-12   pop-tarts strawberry  poptarts strawberry
2   POPTART-STRAW   Pop-Tarts Strawberry  poptarts strawberry
3   PT_STRAWBERRY   pop-tarts strawberry  poptarts strawberry
8         BAT-AA4    AA Batteries 4-Pack   4pack aa batteries
9     BATT_AA_4PK    AA Batteries 4-Pack   4pack aa batteries
10       FLASH-01         FLASHLIGHT LED       flashlight led
11      LED-FLASH         FLASHLIGHT LED       flashlight led
12        BEER-24   Light Beer (24-pack)    24pack beer light
13       LTBEER24   Light Beer (24-pack)    24pack beer light
14        BG-MONO    Monopoly Board Game  board game monopoly
15    MONOPOLY-01    Monopoly Board Game  board game monopoly
16      GAME-MONO  Board Game - Monopoly  board game monopoly
17       CARDS-01          Playing Cards        cards playing
18   PLAYCARD-STD          PLAYING CARDS        cards playing
21       SOUP-CAN          Soup - Canned          canned soup
22      

Here we notice that SKU#459812 Strawberry Pop-Tarts was absorbed by one of the three remaining Pop-Tart SKUs. Beyond that, we have 13 groups that hold multiple SKUs. From a data analysis perspective, this emphasis the importance of the data wrangling and cleaning steps prior to any analysis; as each decision made has an affect on any later analysis.

### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [8]:
# TODO
# Check that no canonical SKUs are null
assert cat['canonical_sku'].notna().all(), "Null values found in canonical_sku"
# Check that the number of rows has not changed
assert len(cat) == len(pd.read_csv(PATH)), "Row count changed"
# Ouput
print("Mapping is safe: no nulls and row count unchanged.")

Mapping is safe: no nulls and row count unchanged.


**Q6: Explanation:** From the code chunk, we have that the mapping done is safe, and that the mapping produced no nulls in the canonical column and did not change the number of rows.

### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.


**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.


**a)** Paper towels and AA batteries 4-pack are two additional fragmented products. Due to their product simplicity, I would group them together. A 4-pack of AA Batteries is specific, including only that type and quantity of batters; thus it can be grouped together. While there are different types of paper towels on the market, the two paper towels are listed as just that with no further information, and thus we should be able to group them as well.

**b)** Consolidating the data is never just a data-cleaning task here. From the beginning, their are business implications. Whether grouping by product name or supplier, there is a tradeoff that has to be evaluated. For instance, for the strawberry Pop-Tarts, we were concerned with the specific product; allowing us to essentially disregard the supplier, quanitity in package, cost, and category in our analysis. However, this is not always the case. We could be interested in seeing how the different suppliers of strawberry Pop-Tarts sold at varying locations of Walmart, which would require our data-cleaning process to change. Therefore, we have to keep our business goal in mind throughout the entire consolidation process.

Further, in this catalog, I am unsure with what I would do with the playing cards products. The product name "playing cards" is very broad and can encompas a lot of different cards and card games. Here, I would want to consult the people interested in the analysis to see what information they would want from any analysis. If they were interested in specific vendors or types of card games, that would require more in-depth, less normalized analysis; however, if they wanted general market research, then we have more flexibility to normalize the data.